# Train the priority classifier (v3.0) on Colab

DistilBERT (`distilbert-base-uncased`) fine-tuned on `insanar/prior-mail-priority` (config `v4`, class-balanced ~2000/class).

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- This notebook clones the repo from GitHub; the migration lives on `main`.

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime → T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

2.11.0+cu128 Tesla T4


## 2. Clone the repo

Private org repo → use a GitHub PAT with `repo` scope.

In [ ]:
!git clone https://github.com/PJK-GM095-PIJAK/prior-mail-model.git
%cd prior-mail-model
# !git checkout <branch-or-sha>   # optional: pin a specific commit

Cloning into 'prior-mail-model'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 195 (delta 89), reused 152 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 98.12 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/prior-mail-model/prior-mail-model


## 3. Install dependencies

Not `pip install -e .` — Colab's Python doesn't satisfy the `>=3.11,<3.12` pin. Install the libs directly and put the repo root on `PYTHONPATH`. (`sentencepiece` is no longer needed — DistilBERT uses WordPiece.)

In [ ]:
!pip install -q transformers datasets accelerate evaluate wandb emoji huggingface_hub
%env PYTHONPATH=.

env: PYTHONPATH=.


## 4. Experiment tracking (pick ONE)

Live tracking with W&B (project `priormail`):

In [ ]:
import wandb
wandb.login()   # paste your W&B key

True

...or skip tracking. If so, run **only** the line below in its own cell — with **no trailing comment** (a comment silently breaks `%env`):

In [ ]:
%env WANDB_MODE=offline

env: WANDB_MODE=offline


## 5. Build the dataset

Downloads `insanar/prior-mail-priority:v4` (the class-balanced cut) and writes `data/processed/priority`. **This cell decides the dataset version** — both the trainer and eval just read `data/processed/priority` from disk, so whatever you build here is what gets trained and evaluated on. After it runs, check the logged split counts: train should be ~8000 (2000×4), and confirm the val/test splits aren't lopsided (early stopping uses val macro-F1).

In [ ]:
!python -m src.data.prepare --subset v4

python -m src.data.prepare
2026-06-09 00:57:51,164 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-06-09 00:57:51,593 INFO datasets: TensorFlow version 2.20.0 available.
2026-06-09 00:57:51,594 INFO datasets: JAX version 0.7.2 available.
2026-06-09 00:57:52,205 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/insanar/prior-mail-priority/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-09 00:57:52,206 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-09 00:57:52,246 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/insanar/prior-mail-priority/4aae4bd09e54cf2f7f4acfa9120c854990ebf36f/README.md "HTTP/1.1 200 OK"
2026-06-09 00:57:52,330 INFO httpx: HTTP Request: HEAD https://huggingface.co/datasets/insanar/prior-mail-priority/resolve/4aae4bd09e54cf2f7f4acfa9120c854990ebf36f/prior-mail-prio

## 6. Train

DistilBERT on ~8k balanced examples × 4 epochs ≈ a few minutes on a T4. The config requests `bf16`; on a T4 (no bf16) the trainer auto-falls back to `fp16`. `class_weights: balanced` stays on, but on the balanced v4 set the inverse-frequency weights collapse to ≈1.0 — the dataset balance is what now prevents the majority-class (`normal`) collapse.

In [ ]:
!make train config=configs/priority_v3.yaml

## 7. Evaluate against the gates

Promotion gates: macro-F1 ≥ 0.80, per-class recall ≥ 0.65, p95 < 500 ms on CPU.

In [ ]:
!make eval config=configs/priority_v3.yaml

In [ ]:
# ── Step 6b: write model card (required before export) ────────────────────────
import json, os

with open("eval/results/priority/eval_report.json") as f:
    m = json.load(f)["metrics"]

lines = [
    "# Priority Classifier v3.0",
    "",
    "DistilBERT (`distilbert-base-uncased`) fine-tuned on `insanar/prior-mail-priority` (config `v4`, class-balanced ~2000/class).",
    "",
    "## Performance (held-out test set)",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| Macro F1 | {m['macro_f1']:.3f} |",
    f"| Recall — urgent | {m['recall_urgent']:.3f} |",
    f"| Recall — high | {m['recall_high']:.3f} |",
    f"| Recall — normal | {m['recall_normal']:.3f} |",
    f"| Recall — low | {m['recall_low']:.3f} |",
    f"| Inference p95 (CPU) | {m['p95_ms']:.0f} ms |",
    "",
    "## Known limits",
    "- Topic→priority proxy, not true urgency understanding",
    "- Trained on English only",
    "- Dataset includes synthetic examples",
]

os.makedirs("checkpoints/priority_v3", exist_ok=True)
with open("checkpoints/priority_v3/model_card.md", "w") as f:
    f.write("\n".join(lines))

print("✓ model_card.md written")

## 8. Save the checkpoint

Zip + download (Drive is flaky):

In [ ]:
!cd checkpoints && zip -qr priority_v3.zip priority_v3
from google.colab import files
files.download("checkpoints/priority_v3.zip")

Or publish to the HuggingFace Hub (needs a **write** token; the uploader handles the Xet stall itself):

In [ ]:
!cp eval/results/priority/eval_report.json checkpoints/priority_v3/

In [ ]:
from huggingface_hub import logout
logout()


In [ ]:
from huggingface_hub import login
login()   # HF token with write access

In [ ]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("HuggingFace WRITE token: ")


HuggingFace WRITE token: ··········


In [ ]:
!python -m src.exporter.export --checkpoint checkpoints/priority_v3 --hf-org insanar --version v3.0